# Step 0 · Diagnostics of the CPC / IPC codes

For every code in `config/sector.yaml` we count the families it selects and how many of them **mention eyewear** in a title or abstract (any application of the family). The share tells us whether a code is specific to smart glasses (**core**) or generic (**component**: kept only together with eyewear terms or after the semantic filter).

Rules as in the tutorials: DOCDB families, `ipr_type = 'PI'`, no dummy applications, earliest priority year inside the window.

Run it on **TEST** first to check that it works, then on **PROD** for the real numbers.

In [ ]:
%pip install -q pyyaml

In [ ]:
from pathlib import Path

import pandas as pd
from sqlalchemy import distinct, func, literal, select, union_all

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import TLS201_APPLN, TLS209_APPLN_IPC, TLS224_APPLN_CPC

from src.sector import families_by_cpc, families_by_keywords, load_config, scope_filter, symbol_prefix_condition

ENV = "TEST"                       # "TEST" to check the code, "PROD" for the real numbers
OUTPUT_DIR = Path("outputs/step0")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = load_config()
sel = cfg["selection"]
patstat = PatstatClient(env=ENV)
db = patstat.orm()

print(f"{cfg['sector_name']} | env {ENV} | years {cfg['years']['min']}-{cfg['years']['max']}")
print({k: len(v) for k, v in sel.items()})

## 1. Reference sets

- **eyewear text**: families where any application mentions an eyewear term or a specific keyword;
- **core CPC**: families with at least one core CPC code.

In [ ]:
eyewear_text = families_by_keywords(db, cfg, sel["eyewear_terms"] + sel["specific_keywords"]).distinct().subquery()
core_cpc = families_by_cpc(db, cfg, sel["core_cpc"]).distinct().subquery()

## 2. One row per code

All codes go in a single query (`UNION ALL` of one labelled selection per code), so the reference sets are computed once.

In [ ]:
def labelled_selection(label, code, symbol_table, symbol_column):
    """(label, docdb_family_id) pairs for the families with a symbol under the code."""
    return (
        select(literal(label).label("code"), TLS201_APPLN.docdb_family_id.label("docdb_family_id"))
        .select_from(TLS201_APPLN)
        .join(symbol_table, TLS201_APPLN.appln_id == symbol_table.appln_id)
        .where(*scope_filter(cfg), symbol_prefix_condition(symbol_column, [code]))
    )


codes = (
    [("CPC", "core", c) for c in sel["core_cpc"]]
    + [("CPC", "component", c) for c in sel["component_cpc"]]
    + [("IPC", "core", c) for c in sel["core_ipc"]]
    + [("IPC", "component", c) for c in sel["component_ipc"]]
)
selections = [
    labelled_selection(f"{scheme} {c}", c, *((TLS224_APPLN_CPC, TLS224_APPLN_CPC.cpc_class_symbol) if scheme == "CPC"
                                          else (TLS209_APPLN_IPC, TLS209_APPLN_IPC.ipc_class_symbol)))
    for scheme, _, c in codes
]
pairs = union_all(*selections).subquery("pairs")

q = (
    db.query(
        pairs.c.code,
        func.count(distinct(pairs.c.docdb_family_id)).label("families"),
        func.count(distinct(eyewear_text.c.docdb_family_id)).label("with_eyewear_text"),
        func.count(distinct(core_cpc.c.docdb_family_id)).label("also_core_cpc"),
    )
    .select_from(pairs)
    .outerjoin(eyewear_text, eyewear_text.c.docdb_family_id == pairs.c.docdb_family_id)
    .outerjoin(core_cpc, core_cpc.c.docdb_family_id == pairs.c.docdb_family_id)
    .group_by(pairs.c.code)
)
diag = patstat.df(q)

In [ ]:
roles = pd.DataFrame(codes, columns=["scheme", "role", "symbol"])
roles["code"] = roles["scheme"] + " " + roles["symbol"]

diag = roles.merge(diag, on="code", how="left")
diag[["families", "with_eyewear_text", "also_core_cpc"]] = diag[["families", "with_eyewear_text", "also_core_cpc"]].fillna(0).astype(int)
diag["share_eyewear_text"] = (diag["with_eyewear_text"] / diag["families"]).round(3)
diag["share_also_core"] = (diag["also_core_cpc"] / diag["families"]).round(3)
diag = diag.sort_values(["scheme", "role", "share_eyewear_text"], ascending=[True, False, False])

diag.to_csv(OUTPUT_DIR / f"code_diagnostics_{ENV}.csv", index=False)
print(diag.drop(columns="code").to_string(index=False))

## 3. How to read the table

| Pattern | Meaning | Proposal |
|---|---|---|
| high `share_eyewear_text` (e.g. > 0.5) | the code is mostly about eyewear | **core** |
| medium share, many families | eyewear is one of many applications | **component** (needs eyewear terms or the semantic filter) |
| very low share, very many families (e.g. `G06T7/70`) | generic technology, mostly not eyewear | component only through the semantic filter, or drop |
| high `share_also_core` | the code adds few new families beyond the core codes | low priority |
| `families = 0` | wrong code, or not present in this environment | check the symbol in the scheme |

The text signal misses families with no English title or abstract, so shares are a lower bound; compare codes with each other rather than reading the absolute value.